In [1]:
import torch
import numpy as np
from environment import CliffWalk   # or InfiniteWalk
from episodes import EpisodeCollection, collect_episodes
from actor import ActorPolicyWrapper, collect_episodes_actor
from train_chunk import train_with_chunks    # The function you implemented
from belief_rnn import BeliefRNN, RewardReadout, NextLatentPredictor, ObsReadout, ValueReadout, ActorReadout, QReadout



In [2]:
# --------------------------
# 1. Hyperparameters
# --------------------------

RNN_HIDDEN = 64
INITIAL_CHUNKS = 1
EPISODES_PER_CHUNK = 100
NUM_NEW_CHUNKS = 100

GAMMA = 0.9

# Training step counts per chunk
ACTOR_STEPS  = 10
CRITIC_STEPS = 50
WORLD_STEPS  = 50

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
# --------------------------
# 2. Initialize Environment
# --------------------------
env = CliffWalk(
    n = 2, 
    m = 5, 
    self_transition_prob=0.1,
    gamma=GAMMA,
    generic_reward=-1.0,
    cliff_reward = -10.0,
    target_reward=10.0,
)  



# --------------------------
# 3. Collect Initial Chunks Using a Random Policy
# --------------------------
# Use a dummy policy for initial data collection (uniform over actions)
def random_policy_wrapper(env):
    class RandomPolicy:
        def reset(self): pass
        def __call__(self, obs, prev_action):
            action = np.random.choice(env.action_space)
            return action, 0.0
    return RandomPolicy()

initial_chunks = []
for _ in range(INITIAL_CHUNKS):
    eps = collect_episodes_actor(env, actor_policy=random_policy_wrapper(env), num_episodes=500)
    EC = EpisodeCollection(eps)
    initial_chunks.append(EC)

In [4]:
# --------------------------
# 4. Initialize Models
# --------------------------
input_dim = initial_chunks[0].H      # history dimension = O + A
obs_dim   = initial_chunks[0].O      # observation dimension
num_actions = initial_chunks[0].A    # action dimension

belief_model = BeliefRNN(input_dim=input_dim, latent_dim=RNN_HIDDEN)
value_model  = ValueReadout(RNN_HIDDEN)
q_model      = QReadout(RNN_HIDDEN, 4)
reward_model = RewardReadout(RNN_HIDDEN)
pred_model   = NextLatentPredictor(RNN_HIDDEN)
obs_model    = ObsReadout(RNN_HIDDEN, obs_dim)
actor_model  = ActorReadout(latent_dim=RNN_HIDDEN, num_actions=num_actions, hidden_dim=64)

# Move to device
for m in (belief_model, value_model, q_model, reward_model, pred_model, obs_model, actor_model):
    m.to(DEVICE)

# Combine models into a single tuple for training
models = (
    belief_model,
    actor_model,
    value_model,
    q_model,
    reward_model,
    pred_model,
    obs_model
)

In [5]:
# --------------------------
# 5. Run Training Loop
# --------------------------
models = train_with_chunks(
    env=env,
    models=models,
    initial_chunks=initial_chunks,
    num_new_chunks=NUM_NEW_CHUNKS,
    episodes_per_chunk=EPISODES_PER_CHUNK,
    gamma=GAMMA,
    actor_steps=ACTOR_STEPS,
    world_steps=WORLD_STEPS,
    lambda_actor=1.0,
    lambda_value=1.0,
    lambda_world=1.0,
    device=DEVICE
)

print("Training complete!")

[Chunk 0] success=0.02, cliff=0.98, mean_return=-9.931, mean_len=7.950
actor=-0.003, value=15.323, world=32.461, 
[Chunk 1] success=0.02, cliff=0.98, mean_return=-9.902, mean_len=6.830
actor=-0.023, value=4.657, world=27.262, 
[Chunk 2] success=0.01, cliff=0.99, mean_return=-9.954, mean_len=8.080
actor=-0.054, value=1.871, world=24.085, 
[Chunk 3] success=0.02, cliff=0.98, mean_return=-9.818, mean_len=7.290
actor=-0.054, value=2.044, world=19.483, 
[Chunk 4] success=0.03, cliff=0.97, mean_return=-9.920, mean_len=7.920
actor=-0.042, value=2.902, world=13.823, 
[Chunk 5] success=0.04, cliff=0.96, mean_return=-9.741, mean_len=8.850
actor=-0.052, value=3.403, world=11.259, 
[Chunk 6] success=0.00, cliff=1.00, mean_return=-10.000, mean_len=9.180
actor=-0.046, value=0.225, world=9.408, 
[Chunk 7] success=0.02, cliff=0.98, mean_return=-9.843, mean_len=8.830
actor=-0.060, value=2.060, world=8.139, 
[Chunk 8] success=0.02, cliff=0.98, mean_return=-9.922, mean_len=8.630
actor=-0.159, value=2.522